# Topology Ablation — Encoding × Topology 2×2 (controlled)

**The foundational controlled experiment.** Tests whether the basin one-hot encoding makes hand-designed topology features redundant — the confound that plausibly explains every negative result so far.

All four conditions run on **stock NeuralHydrology `cudalstm`** — identical, well-tuned trainer; the ONLY differences are two config flags. No custom model code, no architecture confound, GPU-native and fast.

|  | topology OFF | topology ON |
|---|---|---|
| **one-hot ON** | L | L+T |
| **one-hot OFF** | L_noID | L_noID+T |

**Headline contrast:** `(L_noID+T) − L_noID` — does topology help when the model *cannot* memorize basin identity? Predict **> 0**. Compare to `(L+T) − L` (predict ≈ 0). If the prediction holds, the paper's thesis becomes *'network structure helps streamflow LSTMs only in the can't-memorize regime'* (cf. Kipf-Welling GCN theory).

## How to use
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Runtime → **Run all**.
3. Outputs save to Drive; final cell prints the 2×2 + contrasts. Idempotent.

## Cell 1 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 2 — Config

In [2]:
import os
GITHUB_URL = 'https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH = ''
# Networks to run the 2x2 on. component0 = full 183-basin; the sg_* are small local subgraphs.
NETWORKS = ['component0', 'sg_northeast', 'sg_ohio']
SEED = 11   # single seed for now; multi-seed only at publication time

AUTO = ['/content/drive/MyDrive/datasets/camels_us',
        '/content/drive/MyDrive/neural_hydro/datasets/camels_us',
        '/content/drive/MyDrive/neural_hydrology/datasets/camels_us',
        '/content/drive/MyDrive/camels_us', '/content/drive/MyDrive/data/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH = c; print('Found camels_us at', c); break
    else: raise RuntimeError('Set DRIVE_CAMELS_PATH')
assert os.path.isfile(os.path.join(DRIVE_CAMELS_PATH, 'camels_attributes_v2.0', 'camels_topo.txt'))
DRIVE_RUNS = '/content/drive/MyDrive/neural_hydrology_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('Networks:', NETWORKS, ' seed', SEED)

Found camels_us at /content/drive/MyDrive/camels_us
Networks: ['component0', 'sg_northeast', 'sg_ohio']  seed 11


## Cell 3 — Clone repo

In [3]:
REPO_DIR = '/content/nh'
import shutil
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 3

Cloning into '/content/nh'...
remote: Enumerating objects: 5794, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 5794 (delta 27), reused 77 (delta 20), pack-reused 5704 (from 2)
Receiving objects: 100% (5794/5794), 805.47 MiB | 22.85 MiB/s, done.
Resolving deltas: 100% (547/547), done.
Updating files: 100% (4994/4994), done.
/content/nh
1230899 (HEAD -> main, origin/main, origin/HEAD) fix topology_2x2 notebook Cell 5: split 'if ...: !rm' one-liners
3ebfd33 topology_ablation: controlled restart — encoding × topology 2×2 on stock NH
87233c7 local_subgraphs: promote Colab GPU notebook to primary path


## Cell 4 — Install deps (numpy<2 pin)

In [4]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import importlib, sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__}  pandas {pd.__version__}  torch {torch.__version__}  CUDA {torch.cuda.is_available()}')

/content/nh
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


/tmp/ipykernel_7498/2807677845.py:7: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy as np, pandas as pd, torch


numpy 1.26.4  pandas 2.1.4  torch 2.11.0+cu128  CUDA True


## Cell 5 — Symlink data + runs

In [5]:
%cd {REPO_DIR}
import shutil
RD = os.path.join(REPO_DIR, 'datasets', 'camels_us')
os.makedirs(os.path.dirname(RD), exist_ok=True)
if os.path.islink(RD):
    os.unlink(RD)
elif os.path.isdir(RD):
    shutil.rmtree(RD, ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR = os.path.join(REPO_DIR, 'runs')
if os.path.islink(RR):
    os.unlink(RR)
elif os.path.isdir(RR):
    shutil.rmtree(RR, ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR, 'runs', 'topology_ablation'), exist_ok=True)
print('datasets/camels_us ->', DRIVE_CAMELS_PATH)
print('runs/ ->', DRIVE_RUNS)
!ls datasets/camels_us | head -3

/content/nh
datasets/camels_us -> /content/drive/MyDrive/camels_us
runs/ -> /content/drive/MyDrive/neural_hydrology_runs
basin_mean_forcing
camels_attributes_v2.0
usgs_streamflow


## Cell 6 — GPU check

In [6]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

GPU 0: Tesla T4 (UUID: GPU-02f0277d-73d5-293f-9f79-0fbac30a5e2c)
GPU: Tesla T4


## Cell 7 — Generate the topology static-attributes file

Writes `camels_attributes_v2.0/camels_topology.txt` (network-position features per basin). NH auto-loads it as static attributes. Idempotent — safe to re-run.

In [7]:
%cd {REPO_DIR}
!python experiments/topology_ablation/generate_topology_attributes.py 2>&1 | tail -8

/content/nh

Use in a config via:
  static_attributes:
    - graph_depth
    - n_upstream
    - total_upstream_area
    - in_degree
    - frac_upstream_area


## Cell 8 — Run the 2×2 on each network (stock cudalstm, single seed)

In [11]:
%cd {REPO_DIR}
nets = ' '.join(NETWORKS)
!python experiments/topology_ablation/run_2x2.py --networks {nets} --seed {SEED} --device cuda:0 --epochs 30

/content/nh

=== configs component0 seed=11 ===
Wrote 4 configs for network=component0 seed=11:
  L_component0_seed11.yaml
  L_T_component0_seed11.yaml
  L_noID_component0_seed11.yaml
  L_noID_T_component0_seed11.yaml

=== L component0 seed=11 TRAIN ===
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than

## Cell 9 — Analyze: the 2×2 + the headline contrast

In [9]:
%cd {REPO_DIR}
!python experiments/topology_ablation/analyze_2x2.py 2>&1 | tail -40
print('\nFull writeup: experiments/topology_ablation/analysis/RESULTS.md')

/content/nh

Wrote table + contrasts + RESULTS.md to /content/nh/experiments/topology_ablation/analysis

Full writeup: experiments/topology_ablation/analysis/RESULTS.md


In [10]:
import glob
print(glob.glob('/content/nh/runs/topology_ablation/component0/*'))
!ls -la /content/nh/runs/topology_ablation/component0/ 2>/dev/null
!nvidia-smi | grep -A1 python


['/content/nh/runs/topology_ablation/component0/L_component0_seed11_2106_040655']
total 4
drwx------ 4 root root 4096 Jun 21 04:32 L_component0_seed11_2106_040655


## Done

The number that matters: **topo benefit WITHOUT one-hot** (`L_noID+T − L_noID`). If it's clearly positive while **topo benefit WITH one-hot** is ≈ 0, the redundancy hypothesis is confirmed and the paper has its controlled, theory-grounded thesis. Pull `experiments/topology_ablation/analysis/RESULTS.md` and ping `crs interpret topology 2x2`.